# Module 3.6 — Design Patterns in Python

### Python Mastery · Track 3: Object-Oriented Programming & Design Patterns

Design patterns are reusable solutions to recurring design problems. They are grouped into creational (how objects are made), structural (how objects are composed), and behavioural (how objects interact). This module presents the most useful patterns in each group, and shows how Python's dynamic features often make them lighter than in other languages.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Identify the problem each pattern solves before studying the code.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Apply creational patterns: Singleton, Factory, and Builder.
2. Apply structural patterns: Adapter, Decorator, and Facade.
3. Apply behavioural patterns: Observer, Strategy, and State.
4. Recognise when a pattern is appropriate and when simpler code suffices.
5. Use first-class functions to simplify several patterns.

**Prerequisites:** Tracks 1 and 2, and Modules 3.1 to 3.5.

---

## Part 1 · Creational Patterns

Creational patterns control how objects are created.

The **Singleton** ensures a class has only one instance, shared everywhere; it suits a single configuration or connection pool. The **Factory** centralises the decision of which class to instantiate behind one function or method, so callers do not hard-code concrete classes. The **Builder** constructs a complex object step by step, which is clearer than a constructor with many arguments.

In [ ]:
# Singleton via __new__: return the same instance every time.
class Config:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.settings = {}        # initialise once
        return cls._instance

a = Config()
b = Config()
a.settings["theme"] = "dark"
print("same instance:", a is b)
print("shared state:", b.settings)

In [ ]:
# Factory: a function decides which concrete class to build from an input.
class Dog:
    def speak(self): return "Woof"
class Cat:
    def speak(self): return "Meow"

def animal_factory(kind):
    """Return an instance of the right class for the given kind."""
    registry = {"dog": Dog, "cat": Cat}
    try:
        return registry[kind]()
    except KeyError:
        raise ValueError(f"unknown animal: {kind}")

for kind in ["dog", "cat"]:
    print(kind, "->", animal_factory(kind).speak())

In [ ]:
# Builder: assemble a complex object step by step with a fluent interface.
class Pizza:
    def __init__(self):
        self.toppings = []
        self.size = "medium"
    def __repr__(self):
        return f"Pizza({self.size}, toppings={self.toppings})"

class PizzaBuilder:
    def __init__(self):
        self.pizza = Pizza()
    def set_size(self, size):
        self.pizza.size = size
        return self                 # returning self enables method chaining
    def add(self, topping):
        self.pizza.toppings.append(topping)
        return self
    def build(self):
        return self.pizza

result = PizzaBuilder().set_size("large").add("cheese").add("mushroom").build()
print(result)

## Part 2 · Structural Patterns

Structural patterns compose objects into larger structures.

The **Adapter** wraps an object so it presents the interface a client expects, letting incompatible classes work together. The **Decorator** wraps an object to add behaviour without changing it. The **Facade** offers one simple interface over a complicated subsystem.

In [ ]:
# Adapter: make an existing class fit an expected interface.
class EuropeanSocket:
    def voltage(self):
        return 230

class USDevice:
    """Expects a method named get_volts(), which EuropeanSocket lacks."""
    def run(self, power_source):
        return f"running at {power_source.get_volts()}V"

class SocketAdapter:
    def __init__(self, socket):
        self.socket = socket
    def get_volts(self):                 # translate the interface
        return self.socket.voltage()

device = USDevice()
print(device.run(SocketAdapter(EuropeanSocket())))

In [ ]:
# Decorator (object form): wrap to add behaviour, keeping the same interface.
class Coffee:
    def cost(self):
        return 50
    def description(self):
        return "coffee"

class MilkDecorator:
    def __init__(self, drink):
        self.drink = drink
    def cost(self):
        return self.drink.cost() + 10          # add to the wrapped cost
    def description(self):
        return self.drink.description() + " + milk"

class SugarDecorator:
    def __init__(self, drink):
        self.drink = drink
    def cost(self):
        return self.drink.cost() + 5
    def description(self):
        return self.drink.description() + " + sugar"

order = SugarDecorator(MilkDecorator(Coffee()))   # wrap repeatedly
print(order.description(), "costs", order.cost())

In [ ]:
# Facade: a single simple call hides a multi-step subsystem.
class CPU:
    def start(self): return "CPU started"
class Memory:
    def load(self): return "memory loaded"
class Disk:
    def read(self): return "disk read"

class Computer:
    """A facade over the boot subsystem."""
    def __init__(self):
        self.cpu, self.memory, self.disk = CPU(), Memory(), Disk()
    def boot(self):
        return " -> ".join([self.cpu.start(), self.memory.load(), self.disk.read()])

print(Computer().boot())   # the caller does not deal with the parts directly

## Part 3 · Behavioural Patterns

Behavioural patterns concern how objects communicate.

The **Observer** lets objects subscribe to an event and be notified when it occurs, decoupling the source from its listeners. The **Strategy** makes an algorithm interchangeable at runtime. The **State** lets an object change its behaviour when its internal state changes, as if it changed class.

In [ ]:
# Observer: subscribers are notified when the subject changes.
class Subject:
    def __init__(self):
        self._observers = []
    def subscribe(self, callback):
        self._observers.append(callback)
    def notify(self, event):
        for callback in self._observers:
            callback(event)               # each observer reacts in its own way

log = []
subject = Subject()
subject.subscribe(lambda e: log.append(f"emailed about {e}"))
subject.subscribe(lambda e: log.append(f"logged {e}"))
subject.notify("new order")
print(log)

In [ ]:
# Strategy: select an interchangeable algorithm at runtime.
def by_name(person): return person["name"]
def by_age(person): return person["age"]

class Sorter:
    def __init__(self, strategy):
        self.strategy = strategy          # the strategy is just a function
    def sort(self, data):
        return sorted(data, key=self.strategy)

people = [{"name": "Chen", "age": 30}, {"name": "Asha", "age": 25}]
print("by name:", Sorter(by_name).sort(people))
print("by age:", Sorter(by_age).sort(people))

In [ ]:
# State: behaviour changes with internal state.
class TrafficLight:
    def __init__(self):
        self.state = "red"
    def next(self):
        # The response to next() depends on the current state.
        transitions = {"red": "green", "green": "yellow", "yellow": "red"}
        self.state = transitions[self.state]
        return self.state

light = TrafficLight()
print("start:", light.state)
for _ in range(4):
    print("->", light.next())

## Part 4 · Simplifying Patterns with First-Class Functions

In languages without first-class functions, several patterns require whole class hierarchies. In Python, a function is an object you can pass around, so Strategy is often just "pass a function," and the Command pattern (wrapping an action as an object) can be a stored callable. The cell below shows both reduced to their essence.

In [ ]:
# Strategy as a plain function argument: no class needed.
def apply(values, operation):
    return [operation(v) for v in values]

print("doubled:", apply([1, 2, 3], lambda x: x * 2))
print("squared:", apply([1, 2, 3], lambda x: x ** 2))

# Command as a stored callable: queue actions, run them later, support undo-style logs.
history = []
def make_command(action, *args):
    return lambda: action(*args)          # capture the call for later

def greet(name): return f"hello {name}"
def add(a, b): return a + b

queue = [make_command(greet, "Asha"), make_command(add, 2, 3)]
for command in queue:
    history.append(command())             # execute each stored command
print("results:", history)

> **Guidance.** Patterns are vocabulary, not goals. Reach for one when it clarifies a real problem of flexibility or decoupling. For straightforward needs, a plain function or a small class is usually better than a formal pattern. The aim is readable, maintainable code.

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — A factory function (Easy)

In [ ]:
def shape_factory(kind, size):
    shapes = {
        "square": lambda s: s * s,
        "circle": lambda s: 3.14159 * s * s,
    }
    return shapes[kind](size)

print("square area:", shape_factory("square", 4))
print("circle area:", round(shape_factory("circle", 4), 2))

### Example 2 — Strategy as a function (Easy)

In [ ]:
def discount_none(price): return price
def discount_ten(price): return price * 0.9

def checkout(price, strategy):
    return strategy(price)

print("no discount:", checkout(100, discount_none))
print("ten percent:", checkout(100, discount_ten))

### Example 3 — A builder with chaining (Medium)

In [ ]:
class Query:
    def __init__(self):
        self.parts = {"select": "*", "table": None, "where": None}
    def select(self, columns):
        self.parts["select"] = columns
        return self
    def from_(self, table):
        self.parts["table"] = table
        return self
    def where(self, condition):
        self.parts["where"] = condition
        return self
    def build(self):
        q = f"SELECT {self.parts['select']} FROM {self.parts['table']}"
        if self.parts["where"]:
            q += f" WHERE {self.parts['where']}"
        return q

sql = Query().select("name, age").from_("users").where("age > 18").build()
print(sql)

### Example 4 — Observer for a simple event system (Medium)

In [ ]:
class EventBus:
    def __init__(self):
        self.handlers = {}
    def on(self, event, handler):
        self.handlers.setdefault(event, []).append(handler)
    def emit(self, event, data):
        for handler in self.handlers.get(event, []):
            handler(data)

bus = EventBus()
received = []
bus.on("login", lambda user: received.append(f"welcome {user}"))
bus.on("login", lambda user: received.append(f"audit: {user} logged in"))
bus.emit("login", "Asha")
print(received)

### Example 5 — Stacked decorators on an object (Difficult)

In [ ]:
class TextStream:
    def write(self, text):
        return text

class UpperCase:
    def __init__(self, stream):
        self.stream = stream
    def write(self, text):
        return self.stream.write(text.upper())

class Bracketed:
    def __init__(self, stream):
        self.stream = stream
    def write(self, text):
        return self.stream.write(f"[{text}]")

# Wrap in layers; each adds one transformation. Order of wrapping matters.
stream = Bracketed(UpperCase(TextStream()))
print(stream.write("hello"))          # [HELLO]

stream2 = UpperCase(Bracketed(TextStream()))
print(stream2.write("hello"))         # [HELLO] uppercased -> [HELLO]

### Example 6 — A state machine for an order (Difficult)

In [ ]:
class Order:
    """An order whose allowed actions depend on its state."""
    def __init__(self):
        self.state = "new"
        self.history = ["new"]

    def advance(self):
        transitions = {
            "new": "paid",
            "paid": "shipped",
            "shipped": "delivered",
            "delivered": "delivered",     # terminal state stays put
        }
        self.state = transitions[self.state]
        self.history.append(self.state)
        return self.state

    def can_cancel(self):
        # Behaviour depends on state: only cancellable before shipping.
        return self.state in {"new", "paid"}

order = Order()
print("can cancel now:", order.can_cancel())
order.advance()      # paid
order.advance()      # shipped
print("can cancel after shipping:", order.can_cancel())
print("history:", order.history)

---

## Exercises

**Exercise 1 (Easy).** Write a factory function `make_greeter(language)` that returns a function which greets by name in the chosen language (`"en"` -> "Hello, X"; `"fr"` -> "Bonjour, X"). Demonstrate both.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Implement the Strategy pattern with plain functions: write `summarise(numbers, strategy)` and call it with a `sum`-based strategy and a `max`-based strategy.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Implement a Singleton class `Settings` (using `__new__`) so that every instance shares the same dictionary `data`. Set a value through one reference and read it through another.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Build a simple Observer: a `Newsletter` class where subscribers (functions) are added with `subscribe` and notified with `publish(issue)`. Demonstrate two subscribers receiving an issue.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Implement a small state machine `Turnstile` with states `"locked"` and `"unlocked"`. A `coin()` action unlocks it; a `push()` action locks it again and reports `"passed"` only if it was unlocked. Demonstrate a full cycle.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
def make_greeter(language):
    templates = {"en": "Hello, {}", "fr": "Bonjour, {}"}
    template = templates[language]
    return lambda name: template.format(name)

print(make_greeter("en")("Asha"))
print(make_greeter("fr")("Ben"))

**Solution 2**

In [ ]:
def summarise(numbers, strategy):
    return strategy(numbers)

print("sum:", summarise([1, 2, 3, 4], sum))
print("max:", summarise([1, 2, 3, 4], max))

**Solution 3**

In [ ]:
class Settings:
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.data = {}
        return cls._instance

x = Settings()
y = Settings()
x.data["lang"] = "en"
print("shared:", y.data, "| same object:", x is y)

**Solution 4**

In [ ]:
class Newsletter:
    def __init__(self):
        self.subscribers = []
    def subscribe(self, fn):
        self.subscribers.append(fn)
    def publish(self, issue):
        for fn in self.subscribers:
            fn(issue)

received = []
n = Newsletter()
n.subscribe(lambda i: received.append(f"reader A got {i}"))
n.subscribe(lambda i: received.append(f"reader B got {i}"))
n.publish("May edition")
print(received)

**Solution 5**

In [ ]:
class Turnstile:
    def __init__(self):
        self.state = "locked"
    def coin(self):
        self.state = "unlocked"
        return self.state
    def push(self):
        if self.state == "unlocked":
            self.state = "locked"
            return "passed"
        return "blocked"

t = Turnstile()
print("push while locked:", t.push())   # blocked
t.coin()
print("push after coin:", t.push())     # passed
print("push again:", t.push())          # blocked

---

## Summary and Key Points

- Creational patterns control object creation: Singleton (one shared instance), Factory (centralised choice of class), and Builder (step-by-step assembly with chaining).
- Structural patterns compose objects: Adapter (translate an interface), Decorator (wrap to add behaviour), and Facade (one simple interface over a subsystem).
- Behavioural patterns govern interaction: Observer (notify subscribers), Strategy (interchangeable algorithms), and State (behaviour that changes with state).
- Python's first-class functions reduce several patterns to passing a function, as with Strategy and Command.
- Patterns are vocabulary for real design problems; prefer the simplest solution and use a pattern only when it adds genuine clarity or flexibility.

### Next module: 3.7 — SOLID Principles & Architecture

The next module covers five principles for maintainable object-oriented design and the practical preference for composition over inheritance.